# SWAP-Stress: the unified training table

Stage 02 (`swapstress-build-table`) joins two things into one table:

- standardized observations — `theta`, `suction_cm`, `depth_cm`, `rosetta_level`
- the per-source Earth Engine feature tables from stage 01

The result is **observation-level**: one row per (theta, suction) pair, with
every covariate attached and `log10_suction_cm` as the target.

1. Stage 02, resolved and loaded
2. Schema and join health
3. Distributions by source
4. Feature groups
5. Where the observations are

Every code cell calls `swapstress` —
`swapstress.features.build_training_table`, `swapstress.features.features`, and
`swapstress.model.data` — rather than restating what those modules do.

In [ ]:
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from swapstress.features import build_training_table
from swapstress.features.build_training_table import default_output_path
from swapstress.sources.registry import DEFAULT_SOURCES

# ---------------------------------------------------------------------------
# The one path you may have to change: where the project data tree is mounted.
# The table path itself comes from the library, so it tracks whatever stage 02
# would write for this scale.
# ---------------------------------------------------------------------------
DATA_ROOT = os.environ.get("SWAPSTRESS_DATA_ROOT", "/nas/soils")
SCALE = "9km_global"

TABLE_PATH = default_output_path(DATA_ROOT, SCALE, embeddings=False)

# Rebuilding reads every standardized CSV and every feature parquet; it takes
# minutes and overwrites TABLE_PATH.
RUN_BUILD_TRAINING_TABLE = False

OUT_DIR = os.path.join("notebooks", "_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

print("DATA_ROOT: ", DATA_ROOT)
print("TABLE_PATH:", TABLE_PATH)

## 1) Stage 02

```bash
uv run swapstress-build-table --data-root /nas/soils
```

The dry run resolves every input the stage reads — each source's feature parquet
and its standardized observation directory — and the table it writes.

In [ ]:
build_training_table.main(["--data-root", DATA_ROOT, "--scale", SCALE, "--dry-run"])

In [ ]:
if RUN_BUILD_TRAINING_TABLE:
    df = build_training_table.build_unified_table(
        sources=DEFAULT_SOURCES,
        data_root=DATA_ROOT,
        output_path=TABLE_PATH,
        scale=SCALE,
    )
else:
    df = pd.read_parquet(TABLE_PATH)
    if df.index.name:
        df = df.reset_index()

print(f"{len(df):,} rows x {df.shape[1]} columns")
df.head()

## 2) Schema and join health

`swapstress.features.features.get_feature_columns` is the same function training
uses to decide what counts as a feature, so the split between identity columns
and the feature block below is the split the model sees.

`swapstress.model.data.audit_dataset` reports observation counts per source and
finds **blocking features** — covariates that are 100 % missing for at least one
source. Those are dropped before training by default, because a feature that is
absent for a whole source lets the model identify the source rather than the
soil.

In [ ]:
from swapstress.features.features import NON_FEATURE_COLS, get_feature_columns
from swapstress.model.data import audit_dataset

feature_cols = get_feature_columns(df, include_depth=False)
print(f"feature columns: {len(feature_cols):,}")

core = [
    c
    for c in [
        "source",
        "sample_id",
        "theta",
        "log10_suction_cm",
        "depth_cm",
        "rosetta_level",
        "lat",
        "lon",
    ]
    if c in df.columns
]
print("core columns present:", core)
print("columns the library never treats as features:", sorted(NON_FEATURE_COLS))

audit = audit_dataset(df, feature_cols)

print("\nobservations by source:")
for source, n in sorted(audit["counts_by_source"].items(), key=lambda kv: -kv[1]):
    print(f"  {source:<12s} {n:>10,d}")

print(
    f"\nblocking features (100% missing for some source): "
    f"{len(audit['blocking_features'])}"
)
for name in audit["blocking_features"][:20]:
    print("  ", name)

In [ ]:
# Per-row feature completeness, by source: a quick read on join problems.
sample = df.sample(n=min(len(df), 500_000), random_state=7)
completeness = sample[feature_cols].notna().mean(axis=1)

(
    pd.DataFrame({"source": sample["source"], "completeness": completeness})
    .groupby("source")["completeness"]
    .agg(["count", "mean", "std", "min"])
    .sort_values("mean")
    .round(4)
)

## 3) Distributions by source

`theta` is the model's one dynamic input and `log10_suction_cm` is its target, so
these two histograms are the training distribution the released product is only
as good as. The lab sources (GSHP, NCSS) reach far drier suctions than the
in-situ sensors (MT Mesonet, LaCADIAN), whose range is capped by the tensiometer.

In [ ]:
plot_cols = [c for c in ["theta", "log10_suction_cm", "depth_cm"] if c in df.columns]
sample = df.sample(n=min(len(df), 300_000), random_state=7)

fig, axes = plt.subplots(1, len(plot_cols), figsize=(5 * len(plot_cols), 4.2), dpi=120)
for ax, col in zip(np.atleast_1d(axes), plot_cols):
    for source, g in sample.groupby("source"):
        vals = pd.to_numeric(g[col], errors="coerce").dropna().to_numpy()
        if vals.size:
            ax.hist(vals, bins=60, alpha=0.4, density=True, label=source)
    ax.set_xlabel(col)
    ax.legend(fontsize=8)
np.atleast_1d(axes)[0].set_ylabel("density")

fig.tight_layout()
out = os.path.join(OUT_DIR, "training_table_distributions.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

## 4) Feature groups

`classify_feature` assigns every column to one of the groups in
`FEATURE_GROUPS`. Those group names are what a train config's `feature_groups`
list selects over, and what `swapstress.model.importance` aggregates permutation
importance into — so this table is the vocabulary the rest of the pipeline uses
to talk about features.

In [ ]:
from swapstress.config import feature_groups_to_exclude
from swapstress.features.features import (
    FEATURE_GROUPS,
    classify_feature,
    filter_feature_groups,
)

groups = pd.Series([classify_feature(c) for c in feature_cols]).value_counts()
print(groups.to_string())

# What the released config keeps: configs/train_9km_global_pruned.toml
released_groups = ["fao", "global_et0", "landsat_bands", "soilgrids", "worldclim"]
kept = filter_feature_groups(feature_cols, feature_groups_to_exclude(released_groups))
print(f"\nthe pruned release keeps {len(kept):,} of {len(feature_cols):,} features")
print("groups in FEATURE_GROUPS:", ", ".join(sorted(FEATURE_GROUPS)))

In [ ]:
by_group = {}
for c in feature_cols:
    by_group.setdefault(classify_feature(c), []).append(c)

wanted = ["nd_mean_gs", "VV_mean", "clay_0-5cm_mean", "elevation"]
present = [w for w in wanted if w in df.columns]
for group in ["landsat_indices", "sentinel1", "soilgrids", "terrain"]:
    if len(present) >= 4:
        break
    if group in by_group and by_group[group][0] not in present:
        present.append(by_group[group][0])

sample = df[present].sample(n=min(len(df), 200_000), random_state=7)

fig, axes = plt.subplots(1, len(present), figsize=(4.5 * len(present), 4), dpi=120)
for ax, col in zip(np.atleast_1d(axes), present):
    vals = pd.to_numeric(sample[col], errors="coerce")
    vals = vals.where(vals > -9990)  # the EE export's unmask sentinel
    ax.hist(vals.dropna().to_numpy(), bins=60, alpha=0.85)
    ax.set_title(f"{col}\n[{classify_feature(col)}]", fontsize=9)

fig.tight_layout()
out = os.path.join(OUT_DIR, "training_table_feature_examples.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

## 5) Where the observations are

A quick scatter of the distinct sample locations. The descriptor's coverage
figure is a different thing — per-pixel valid-retrieval fraction over the
released record — and is rendered by stage 08:

```bash
uv run swapstress-figures --figure coverage
```

In [ ]:
import geopandas as gpd

from swapstress.figures import basemap

# Not part of the packaged basemap assets, which are CONUS-only. Point this at a
# world outline of your choosing, or set it to None to draw the global panel bare.
WORLD_SHP = os.path.join(
    basemap.boundaries_root(),
    "boundaries",
    "world_countries",
    "World_Countries_shp.shp",
)


def conus_axis(ax):
    """Frame a lon/lat axis on CONUS with state outlines. Display plumbing.

    The descriptor's map figures project to Albers and style their own axes;
    this is the quick notebook equivalent, so it corrects the aspect by
    latitude rather than projecting.
    """
    basemap.load_conus_states().boundary.plot(ax=ax, color="0.5", linewidth=0.4)
    ax.set_xlim(*basemap.CONUS_LON)
    ax.set_ylim(*basemap.CONUS_LAT)
    ax.set_aspect(1 / np.cos(np.deg2rad(np.mean(basemap.CONUS_LAT))))
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


pts = df[["sample_id", "source", "lon", "lat"]].drop_duplicates(subset="sample_id")
pts = pts.dropna(subset=["lon", "lat"])
print(f"{len(pts):,} distinct sample locations")

fig, (ax_g, ax_c) = plt.subplots(2, 1, figsize=(11, 9), dpi=130)

if WORLD_SHP and os.path.exists(WORLD_SHP):
    gpd.read_file(WORLD_SHP).to_crs(4326).boundary.plot(
        ax=ax_g, color="0.6", linewidth=0.3
    )
ax_g.set_title("Global")
ax_g.set_xlim(-180, 180)
ax_g.set_ylim(-60, 85)
ax_g.set_xlabel("Longitude")
ax_g.set_ylabel("Latitude")

conus_axis(ax_c)
ax_c.set_title("CONUS")

for source, g in pts.groupby("source"):
    ax_g.scatter(
        g["lon"],
        g["lat"],
        s=3,
        alpha=0.4,
        label=f"{source} (n={len(g):,})",
        rasterized=True,
    )
    ax_c.scatter(g["lon"], g["lat"], s=6, alpha=0.5, label=source, rasterized=True)

ax_g.legend(loc="lower left", fontsize=8, markerscale=3)
ax_c.legend(loc="lower left", fontsize=8, markerscale=3)

fig.tight_layout(h_pad=1.5)
out = os.path.join(OUT_DIR, "training_table_point_maps.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

## Next

This table is stage 03's input:

```bash
uv run swapstress-train --config configs/train_9km_global_pruned.toml \
    --output-dir /nas/soils/swapstress/models/direct_rf_9km_global_pruned
```

`08_validation.ipynb` picks up from the trained model.